# 03 — Anomaly Detection Lab

Explore the trade-off between catching bad events and avoiding false alarms.

We inject synthetic anomalies, then try to find them with
a statistical method (z-score) and an ML method (Isolation Forest).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

np.random.seed(7)
df = pd.read_csv('../data/smart_meter_data.csv', parse_dates=['timestamp'])

# Inject anomalies (same as the demo)
df['is_anomaly_true'] = 0
n = len(df)

theft_idx = np.random.choice(n, size=6, replace=False)
for i in theft_idx:
    span = slice(i, min(i + 8, n))
    df.loc[span, 'load_kw'] *= 0.1
    df.loc[span, 'is_anomaly_true'] = 1

spike_idx = np.random.choice(n, size=6, replace=False)
for i in spike_idx:
    span = slice(i, min(i + 4, n))
    df.loc[span, 'load_kw'] *= 3.5
    df.loc[span, 'is_anomaly_true'] = 1

print(f"Injected {df['is_anomaly_true'].sum()} anomalous points")

### Z-score method — tune the threshold

In [ ]:
window = 24 * 4 * 7
threshold = 3.0  # try changing this to 2.0 or 4.0

roll_mean = df['load_kw'].rolling(window, center=True, min_periods=24).mean()
roll_std = df['load_kw'].rolling(window, center=True, min_periods=24).std()
df['zscore'] = (df['load_kw'] - roll_mean) / roll_std
df['flag_zscore'] = (df['zscore'].abs() > threshold).astype(int)

tp = ((df['flag_zscore'] == 1) & (df['is_anomaly_true'] == 1)).sum()
fp = ((df['flag_zscore'] == 1) & (df['is_anomaly_true'] == 0)).sum()
fn = ((df['flag_zscore'] == 0) & (df['is_anomaly_true'] == 1)).sum()
recall = tp / (tp + fn) if (tp + fn) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
print(f"Z-score (threshold={threshold}): recall={recall:.2f}, precision={precision:.2f}")

### Isolation Forest — tune the contamination

In [ ]:
contamination = 0.02  # try 0.01 or 0.05
features_anom = ['load_kw', 'hour', 'dayofweek', 'temperature_c']

iso = IsolationForest(contamination=contamination, random_state=42)
df['flag_isoforest'] = (iso.fit_predict(df[features_anom]) == -1).astype(int)

tp = ((df['flag_isoforest'] == 1) & (df['is_anomaly_true'] == 1)).sum()
fp = ((df['flag_isoforest'] == 1) & (df['is_anomaly_true'] == 0)).sum()
fn = ((df['flag_isoforest'] == 0) & (df['is_anomaly_true'] == 1)).sum()
recall = tp / (tp + fn) if (tp + fn) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
print(f"Isolation Forest (contamination={contamination}): recall={recall:.2f}, precision={precision:.2f}")

### Visualize a window with anomalies

In [ ]:
anom_indices = df[df['is_anomaly_true'] == 1].index
start = max(anom_indices.min() - 50, 0)
end = min(start + 800, n)
sub = df.iloc[start:end]

plt.figure(figsize=(13, 4))
plt.plot(sub['timestamp'], sub['load_kw'], linewidth=1)
true_pts = sub[sub['is_anomaly_true'] == 1]
plt.scatter(true_pts['timestamp'], true_pts['load_kw'], color='orange', s=40, label='True anomaly')
flagged = sub[sub['flag_isoforest'] == 1]
plt.scatter(flagged['timestamp'], flagged['load_kw'], facecolors='none', edgecolors='red', s=90, label='Flagged by IF')
plt.legend()
plt.tight_layout()
plt.show()

### Your turn

1. Change the z-score threshold to 2.0. What happens to recall? To precision?
2. Change Isolation Forest's `contamination` to 0.005. Does it catch fewer or more anomalies?
3. Try removing `temperature_c` from the feature list. Does detection get worse?
4. Can you find a threshold that gives both recall and precision > 0.7?